<a href="https://colab.research.google.com/github/ZikrullaRaxmatov/ML_DL_projects/blob/main/poseDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mediapipe

In [13]:
import mediapipe as mp
import cv2
import math
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as pltImg

In [26]:
class poseDetector:
  def __init__(self, static_image_mode = False, model_complexity = 1, smooth_landmarks = True, enable_segmentation = False,
               smooth_segmentation = True, min_detection_confidence = 0.5,  min_tracking_confidence = 0.5):
    self.static_image_mode = static_image_mode
    self.model_complexity = model_complexity
    self.smooth_landmarks = smooth_landmarks
    self.enable_segmentation = enable_segmentation
    self.smooth_segmentation = smooth_segmentation
    self.min_detection_confidence = min_detection_confidence
    self.min_tracking_confidence = min_tracking_confidence

    self.mpDraw = mp.solutions.drawing_utils
    self.mpPose = mp.solutions.pose
    self.pose = self.mpPose.Pose(self.static_image_mode, self.model_complexity, self.smooth_landmarks, self.enable_segmentation,
                                 self.smooth_segmentation, self.min_detection_confidence, self.min_tracking_confidence)

  def findPose(self, img, draw = True):
    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    self.results = self.pose.process(imgRGB)

    if self.results.pose_landmarks:
      if draw:
        self.mpDraw.draw_landmarks(img, self.results.pose_landmarks, self.mpPose.POSE_CONNECTIONS)

    return img


  def findPosition(self, img, draw = True):
    self.lmList = []
    if self.results.pose_landmarks:
      for id, lm in enumerate(self.results.pose_landmarks.landmark):
        h, w, c = img.shape
        cx, cy = int(lm.x*w), int(lm.y*h)
        self.lmList.append([id, cx, cy])
        if draw:
          cv2.circle(img, (cx, cy), 3, (0, 255, 0), 3)

    return self.lmList


  def getAngle(self, img, p1, p2, p3):
    x1, y1 = self.lmList[p1][1:]
    x2, y2 = self.lmList[p2][1:]
    x3, y3 = self.lmList[p3][1:]

    angle = math.degrees(math.atan2(y3 - y2, x3 - x2) - math.atan2(y1 - y2, x1 - x2))

    cv2.line(img, (x1, y1), (x2, y2), (255, 255, 255), 2)
    cv2.line(img, (x3, y3), (x2, y2), (255, 255, 255), 2)
    cv2.circle(img, (x1, y1), 10, (0, 0, 255), 2)
    cv2.circle(img, (x1, y1), 7, (0, 0, 255), cv2.FILLED)
    cv2.circle(img, (x2, y2), 10, (0, 0, 255), 2)
    cv2.circle(img, (x2, y2), 7, (0, 0, 255), cv2.FILLED)
    cv2.circle(img, (x3, y3), 10, (0, 0, 255), 2)
    cv2.circle(img, (x3, y3), 7, (0, 0, 255), cv2.FILLED)
    #cv2.putText(img, str(int(angle)), (x2 - 50 , y2 + 5), 1, cv2.FONT_HERSHEY_PLAIN, (0, 0, 0), 2)

    return angle

def main():
  detector = poseDetector()
  input_video_path = 'lifting_gantels.mp4'
  output_video_path = f'pose_{os.path.basename(input_video_path)}'

  cap = cv2.VideoCapture(input_video_path)
  w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), 30, (w, h))

  count = 0
  dir = 0

  while True:

    ret, img = cap.read()

    if not ret:
      print('Video has edned or a error went')
      break

    img = detector.findPose(img, False)
    lmlist = detector.findPosition(img, False)
    if len(lmlist) != 0:
      angle = detector.getAngle(img, 12, 14, 16)
      per = np.interp(angle, (60, 120), (0, 100))
      #print(angle, per)

      if per == 100:
        if dir == 0:
          count += 0.5
          dir = 1

      if per == 0:
        if dir == 1:
          count += 0.5
          dir = 0

      #print(int(count))
    cv2.putText(img, str(int(count)), (50, 400), cv2.FONT_HERSHEY_TRIPLEX, 6, (0, 255, 0), 2)


    out.write(img)

    if cv2.waitKey(1) == 27:
      break

  cap.release()
  out.release()
  cv2.destroyAllWindows()


  '''
  img_path = 'qolga_tayangan_holat.jpg'
  img = cv2.imread(img_path)
  img = detector.findPose(img)
  lmlist = detector.findPosition(img, False)
  detector.getAngle(img, 12, 14, 16)

  cv2.imwrite('edited.jpg', img)
  '''

  #edImg = pltImg.imread('edited.jpg')
  #plt.imshow(edImg)
  #plt.axis('off')
  #plt.show()


if __name__ == "__main__":
  main()

Video has edned or a error went


In [11]:
import moviepy.editor
moviepy.editor.ipython_display("pose_lifting_gantels.mp4")